In [ ]:
# hybrid_predictor.py
import os
import torch
import torch.nn.functional as F
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader
from PIL import Image
import pickle

# ====== Config ======
DATA_CHECK = "data/check"
TRAIN_ROOT = "data/train"
SSL_MODEL_PATH = "ssl_encoder.pth"
FSL_MODEL_PATH = "fsl_encoder_finetuned.pth"
IMAGE_SIZE = 1024
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ====== Encoder Definitions ======
class ResNetEncoder(torch.nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights=None)
        base.fc = torch.nn.Identity()
        self.encoder = torch.nn.Sequential(*list(base.children()))
    def forward(self, x):
        for layer in self.encoder:
            x = layer(x)
        return x.mean(dim=[2, 3])  # global average pooling

# ====== Load SSL Encoder ======
ssl_encoder = ResNetEncoder()
ssl_state = torch.load(SSL_MODEL_PATH, map_location=DEVICE)
ssl_encoder.load_state_dict(ssl_state)
ssl_encoder.to(DEVICE)
ssl_encoder.eval()

# ====== Load FSL Encoder ======
fsl_encoder = ResNetEncoder()
fsl_state = torch.load(FSL_MODEL_PATH, map_location=DEVICE)
fsl_encoder.load_state_dict(fsl_state)
fsl_encoder.to(DEVICE)
fsl_encoder.eval()

# ====== Transform ======
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ====== Build Class Prototypes ======
train_dataset = datasets.ImageFolder(root=TRAIN_ROOT, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=False)

features_by_class = {}
with torch.no_grad():
    for imgs, labels in train_loader:
        imgs = imgs.to(DEVICE)
        ssl_feats = ssl_encoder(imgs)
        fsl_feats = fsl_encoder(imgs)
        hybrid_feats = (ssl_feats + fsl_feats) / 2.0
        for f, label in zip(hybrid_feats, labels):
            label_name = train_dataset.classes[label]
            if label_name not in features_by_class:
                features_by_class[label_name] = []
            features_by_class[label_name].append(f.cpu())

prototypes = {cls: torch.stack(feats).mean(0) for cls, feats in features_by_class.items()}
class_names = list(prototypes.keys())
print(f"✅ Loaded prototypes for hybrid model: {class_names}")

# ====== Save for Analysis ======
with open("hybrid_features.pkl", "wb") as f:
    pickle.dump({
        "class_prototypes": prototypes,
        "class_names": class_names
    }, f)
print("💾 Saved hybrid_features.pkl for feature analysis")

# ====== Prediction Function ======
def predict_image(img_path):
    img = Image.open(img_path).convert("RGB")
    x = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        ssl_feat = ssl_encoder(x)
        fsl_feat = fsl_encoder(x)
        hybrid_feat = (ssl_feat + fsl_feat) / 2.0
        dists = torch.stack([F.pairwise_distance(hybrid_feat, proto.unsqueeze(0)) for proto in prototypes.values()])
        pred_idx = torch.argmin(dists).item()
        pred_class = class_names[pred_idx]
        conf = torch.softmax(-dists, dim=0)[pred_idx].item()
    return pred_class, conf

# ====== Predict on data/check ======
print("\n--- Hybrid Model Predictions ---")
for filename in os.listdir(DATA_CHECK):
    if filename.lower().endswith((".png", ".jpg", ".jpeg")):
        path = os.path.join(DATA_CHECK, filename)
        pred, conf = predict_image(path)
        print(f"{filename}: {pred} (confidence: {conf:.3f})")


RuntimeError: Error(s) in loading state_dict for ResNetEncoder:
	Unexpected key(s) in state_dict: "projector.0.weight", "projector.0.bias", "projector.2.weight", "projector.2.bias". 